# 3 비교모델 결합기 조건 24회 — EfficientAT · PANNs · CED × (지도만 4 · 결합기 한 라벨 4)

**L4 로 실행하십시오.** [셀 1] 이 Blackwell(sm_100·sm_120)이면 멈춥니다. GPU 가 없으면 CPU 로 돕니다(시간 제한 없음).

**'모두 실행' 한 번으로 끝까지 돕니다.** 세션이 끊기면 다시 '모두 실행' — 끝난 회차는 건너뜁니다.
다른 노트북과 동시에 시작해도 됩니다.

| 순서 | 조건 | 회수 | 기다리는 것 |
|---|---|---|---|
| 1 | 지도만(`sup`) — 정답 전량·1,000·500·200 · seed 1 | 3 × 4 | 없음(바로 시작) |
| 2 | 결합기 한 라벨(`plc`) — 같은 4종 · seed 1 | 3 × 4 | Mark5.3 의 `pseudo_labels_cmbone_l<n|all>_s1.csv` 가 지금 교사의 최신 판으로 나올 때까지 |

- `plc` 는 Mark5.3 폴더의 결합기 CSV 를 복사하지 않고 직접 읽습니다(`common/mark53_labels.py`). 학습 전에 교사 → 점수 캐시 → 결합기 CSV 해시 사슬을 대조하고, 어긋나면 멈춥니다.
- 2026-10-22 결정대로 비교모델은 조건마다 1회(seed 1)입니다. 옵티마이저는 Prodigy(run_tag `_pd`).
- 오디오는 `Mark5.2` 폴더의 것을 씁니다(`--data_root`). 교사 데이터 재배치와 무관합니다.
- 논문 표에 비교모델 행을 붙이는 일은 `2 Mark5.3 본실험 70회` 노트북의 마지막 셀이 이 24회가 끝나길 기다렸다가 합니다.

In [ ]:
#========================================================================
# [셀 1] 런타임 확인 — L4 권장 · Blackwell(sm_100·120)이면 멈춤 · CPU 는 허용(멈추지 않음)
#========================================================================
import subprocess, sys
_probe = subprocess.run([sys.executable, "-c",
    "import os, torch\n"
    "ok = torch.cuda.is_available()\n"
    "cap = torch.cuda.get_device_capability(0) if ok else None\n"
    "print('GPU', torch.cuda.get_device_name(0) if ok else '없음 — CPU 로 돈다')\n"
    "print('SM', cap[0] * 10 + cap[1] if cap else 'cpu')\n"
    "print('torch', torch.__version__, '| CUDA', torch.version.cuda, '| CPU 코어', os.cpu_count())"],
    capture_output=True, text=True)
print(_probe.stdout, _probe.stderr[-2000:])
_sm = next((l.split()[1] for l in _probe.stdout.splitlines() if l.startswith("SM ")), "cpu")
if _sm in ("100", "120"):
    raise SystemExit("[STOP] Blackwell(sm_100/sm_120) 런타임입니다. L4 로 바꿔 다시 '모두 실행' 하십시오(2026-10-17 G4 계산 오류).")
!nvidia-smi --query-gpu=name,driver_version,compute_cap,memory.total --format=csv 2>/dev/null || echo "nvidia-smi 없음(CPU 런타임)"
!free -g | head -2

In [ ]:
#========================================================================
# [셀 2] 구글 드라이브 마운트
#========================================================================
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#========================================================================
# [셀 3] 코드 판 확인 + 데이터 목록 — Mark5.2 의 목록과 같게 맞추고 행 수 확인
#========================================================================
import csv, json, os, re, time
EXP = "/content/drive/MyDrive/NewProject/codes/experiments"
DATA_ROOT = f"{EXP}/Mark5.2"
REPOS = {"Baseline_EfficientAT_mn10": "efficientat_mn10", "Baseline_PANNs_CNN6": "panns_cnn6", "Baseline_CED_Tiny": "ced_tiny"}
NEED = {"run_experiments.py": ["CMB_PLAN = [", "pseudo_csv_path, check_fresh"],
        "common/mark53_labels.py": ["def check_fresh", "MARK53_ROOT"],
        "common/naming.py": ['OPTIMIZER_NAME = "prodigy"'],
        "aggregate_baseline.py": ['"plc"']}
for repo in REPOS:
    root = f"{EXP}/{repo}"
    miss = [(f, s) for f, ss in NEED.items() for s in ss
            if not os.path.isfile(f"{root}/{f}") or s not in open(f"{root}/{f}", encoding="utf-8").read()]
    if miss:
        raise SystemExit(f"[STOP] {repo} 코드가 10-23 판이 아닙니다: {miss}")
    for name in ("dataset_labeled.csv", "dataset_unlabeled.csv", "dataset_val.csv", "dataset_test.csv"):
        src, dst = f"{DATA_ROOT}/{name}", f"{root}/{name}"
        if open(src, "rb").read() != (open(dst, "rb").read() if os.path.isfile(dst) else b""):
            open(dst, "wb").write(open(src, "rb").read())
            print(f"[복사] {repo}/{name} <- Mark5.2")
    rows = {n: sum(1 for _ in csv.DictReader(open(f"{root}/{n}", encoding="utf-8-sig")))
            for n in ("dataset_labeled.csv", "dataset_unlabeled.csv", "dataset_val.csv", "dataset_test.csv")}
    if rows != {"dataset_labeled.csv": 2000, "dataset_unlabeled.csv": 2000, "dataset_val.csv": 430, "dataset_test.csv": 430}:
        raise SystemExit(f"[STOP] {repo} 데이터 목록 행 수 {rows}")
    print(f"{repo}: 코드 판 확인 · 데이터 목록 {rows}")
if not os.path.isdir(f"{EXP}/Mark5.3/model"):
    raise SystemExit("[STOP] Mark5.3 폴더가 없습니다 — plc 가 그 폴더의 결합기 CSV 를 읽습니다")

In [ ]:
#========================================================================
# [셀 4] 계획 확인 — 저장소마다 8줄(sup 4 · plc 4), 태그는 Prodigy(_pd_)
#========================================================================
import subprocess


def plan(repo):
    p = subprocess.run(f"{sys.executable} run_experiments.py --preset cmb --data_root {DATA_ROOT} --dry_run",
                       shell=True, capture_output=True, text=True, cwd=f"{EXP}/{repo}")
    rows = [l.strip() for l in p.stdout.splitlines() if "[실행]" in l or "[건너뜀]" in l]
    if p.returncode != 0 or len(rows) != 8 or not all("_pd_" in r or r.endswith("_pd_s1") for r in rows):
        print(p.stdout[-3000:]); print(p.stderr[-3000:])
        raise SystemExit(f"[STOP] {repo} 계획을 읽지 못했습니다(종료 코드 {p.returncode} · {len(rows)}줄, 기대 8줄)")
    return rows


for repo in REPOS:
    rows = plan(repo)
    print(f"{repo}: 남은 {sum('[실행]' in r for r in rows)} / 8")
    for r in rows:
        print("   ", r)

In [ ]:
#========================================================================
# [셀 5] 지도만 ★ 3 × 4회 — 가짜 정답이 필요 없어 바로 돈다. 시간 제한 없음
#========================================================================
for repo in REPOS:
    print(f"\n######## {repo}  sup × 정답 전량·1,000·500·200 · seed 1", flush=True)
    !cd "{EXP}/{repo}" && python run_experiments.py --phase main --conds sup --labeled_ns 0 1000 500 200 --seeds 1 --data_root "{DATA_ROOT}"

In [ ]:
#========================================================================
# [셀 6] Mark5.3 결합기 한 라벨 CSV 4개가 지금 교사의 최신 판으로 나올 때까지 기다린다 — 시간 제한 없음
#========================================================================
sys.path.insert(0, f"{EXP}/Mark5.3/model")
import label_freshness
NEEDED = [f"{EXP}/Mark5.3/pseudo_labels_cmbone_l{l}_s1.csv" for l in ("all", 1000, 500, 200)]


def labels_ready():
    bad = []
    for p in NEEDED:
        try:
            label_freshness.check_pseudo_csv(p, root=f"{EXP}/Mark5.3")
        except (label_freshness.StaleFileError, FileNotFoundError) as e:
            bad.append(f"{os.path.basename(p)}: {str(e)[:120]}")
    return bad


t0 = time.time()
while (bad := labels_ready()):
    print(time.strftime("%H:%M:%S"), f"(경과 {(time.time() - t0) / 60:.0f}분) 결합기 CSV 대기 {len(bad)}/4 — 예: {bad[0]}", flush=True)
    time.sleep(120)
print("결합기 한 라벨 CSV 4개가 지금 교사의 최신 판입니다.")

In [ ]:
#========================================================================
# [셀 7] 결합기 한 라벨 ★ 3 × 4회 — preset cmb(지도만은 끝나 있어 건너뛴다). 시간 제한 없음
#========================================================================
for repo in REPOS:
    print(f"\n######## {repo}  preset cmb (plc 4회)", flush=True)
    !cd "{EXP}/{repo}" && python run_experiments.py --preset cmb --data_root "{DATA_ROOT}"

In [ ]:
#========================================================================
# [셀 8] 확인 — 24회가 전부 끝났는가 (남았으면 멈춘다) · 조건별 test 4지표
#========================================================================
left = {repo: [r for r in plan(repo) if "[실행]" in r] for repo in REPOS}
if any(left.values()):
    for repo, rows in left.items():
        for r in rows:
            print(repo, r)
    raise SystemExit(f"[STOP] 남은 회차 {sum(map(len, left.values()))}개 — 셀 5·7 출력의 [FAIL] 을 확인하고 다시 '모두 실행'")
import pandas as pd
for repo, mtag in REPOS.items():
    subprocess.run([sys.executable, "aggregate_baseline.py", "--split", "test", "--conds", "sup", "plc", "--seeds", "1"],
                   cwd=f"{EXP}/{repo}", check=True)
    raw = pd.read_csv(f"{EXP}/{repo}/plots/aggregate_raw_{mtag}_test.csv")
    cols = [c for c in ("cond", "labeled", "seed", "Accuracy", "Macro Precision", "Macro Recall", "Macro F1") if c in raw.columns]
    display(raw[cols])
print("24회 완료 — 논문 표는 '2 Mark5.3 본실험 70회' 노트북의 마지막 셀이 만듭니다.")